In [3]:
import sys
import os
import pandas as pd

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)


In [4]:
from src.entsoe_client import (
    get_day_ahead_prices,
    get_generation,
    get_wind_solar_forecast,
    get_crossborder_flows
)


ModuleNotFoundError: No module named 'entsoe'

In [ ]:
# Define time range for data extraction

START = "2020-01-01"
END = "2020-01-07"

In [ ]:
# ---------------------------------------------
#  Fetch day-ahead prices
# ---------------------------------------------
prices = get_day_ahead_prices(START, END)
prices = prices.to_frame("price_eur_mwh")
prices = prices.tz_convert("Europe/Vilnius")


In [ ]:
# ---------------------------------------------
#  Fetch actual generation data
# ---------------------------------------------
gen = get_generation(START, END)

# Flatten MultiIndex columns (technology, type)

gen.columns = [
    f"{tech.lower().replace(' ', '_').replace('-', '_')}_{kind.lower().replace(' ', '_')}"
    for tech, kind in gen.columns
]

# Select only required generation columns
gen = gen[[
    "solar_actual_aggregated",
    "wind_onshore_actual_aggregated"
]]

# Rename columns to clearer names
gen = gen.rename(columns={
    "solar_actual_aggregated": "gen_solar_mw",
    "wind_onshore_actual_aggregated": "gen_wind_onshore_mw"
})


In [ ]:
# ---------------------------------------------
#  Fetch wind & solar forecast
# ---------------------------------------------
forecast = get_wind_solar_forecast(START, END)

forecast = forecast.rename(columns={
    "Solar": "fc_solar_mw",
    "Wind Onshore": "fc_wind_onshore_mw"
})

forecast = forecast.tz_convert("Europe/Vilnius")


In [ ]:
# ---------------------------------------------
#  Create unified hourly index
# ---------------------------------------------
idx = pd.date_range(
    START,
    END,
    freq="h",
    tz="Europe/Vilnius",
    inclusive="left"
)


In [ ]:
# ---------------------------------------------
# Reindex all datasets to the same timeline
# ---------------------------------------------
prices = prices.tz_convert("Europe/Vilnius").reindex(idx)
gen = gen.tz_convert("Europe/Vilnius").reindex(idx)
forecast = forecast.reindex(idx)


In [ ]:
# ---------------------------------------------
#  Fetch and align cross-border flows
# ---------------------------------------------
flows = get_crossborder_flows(START, END, tz="Europe/Vilnius")
flows = flows.reindex(idx)


In [ ]:
# ---------------------------------------------
#  Combine all datasets into a single DataFrame
# ---------------------------------------------
market_df = (
    prices
    .join(gen, how="left")
    .join(forecast, how="left")
    .join(flows, how="left")
)


In [ ]:
# ---------------------------------------------
#  Inspect dataset
# ---------------------------------------------
market_df.info()
market_df.head()
market_df.isna().sum()


In [ ]:
# ---------------------------------------------
#  Save dataset to CSV
# ---------------------------------------------
output_path = f"../data/market_{START}_{END}.csv"
market_df.to_csv(output_path)
print("Duomenys išsaugoti į:", output_path)